# dcgan-normal-init-002 — worked example 1: Full DCGAN init: Conv/ConvT from N(0,0.02), BatchNorm weight from N(1,0.02), bias 0

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-normal-init-002`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

The full DCGAN recipe (Radford et al.) initializes every Conv2d / ConvTranspose2d weight from `N(0, 0.02)`. BatchNorm2d layers get a separate convention: their affine `weight` (gamma) is drawn from `N(1.0, 0.02)` and their `bias` (beta) is zeroed. A single `init_fn` dispatched through `model.apply` handles every submodule type recursively.

## Worked solution

**Step 1 — write one dispatcher.** `model.apply(fn)` calls `fn` on every submodule (depth-first, including the root). So we write a single `init_fn(m)` that branches on `type(m)`.

**Step 2 — handle the conv family.** `if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d))`: call `nn.init.normal_(m.weight, 0.0, 0.02)` in place. This is the headline DCGAN move — the 0.02 std keeps early-layer activations from saturating.

**Step 3 — handle BatchNorm separately.** `elif isinstance(m, nn.BatchNorm2d)`: BN's learnable scale `weight` should start near 1 (identity scaling) but with a little jitter, so `nn.init.normal_(m.weight, 1.0, 0.02)`; its `bias` starts at 0, so `nn.init.constant_(m.bias, 0.0)`. Mixing these two up is the classic bug — conv gets mean 0, BN gets mean 1.

**Step 4 — leave everything else alone.** No `else` branch touches Sequential containers or the model root; they fall through untouched.

**Step 5 — verify.** After applying, the conv weight's empirical std should be ~0.02 and the BN weight mean ~1.0. We sample a large conv so the empirical statistics are tight.

In [ ]:
def dcgan_init(model: nn.Module) -> nn.Module:
    def init_fn(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.constant_(m.bias, 0.0)
    model.apply(init_fn)
    return model

t.manual_seed(0)
net = nn.Sequential(
    nn.ConvTranspose2d(100, 64, 4, 1, 0),
    nn.BatchNorm2d(64),
    nn.Conv2d(64, 3, 3, 1, 1),
)
dcgan_init(net)
conv = net[2]
bn = net[1]
print('conv weight std :', round(conv.weight.std().item(), 4))
print('bn weight mean  :', round(bn.weight.mean().item(), 4))
print('bn bias max abs :', round(bn.bias.abs().max().item(), 6))